In [ ]:
"""
dram_chips.py — Simulação de dois chips DRAM com ciclo real RAS/CAS
Python 3.8+

CHIP 1: DRAM 128M × 4 bits  (512 Mbits = 64 MB)
CHIP 2: DRAM 4096K × 1 bit  (4 Mbits  = 0,5 MB / chip)
        Sistema: 32 chips em paralelo → palavra de 32 bits, 16 MB total
"""


# ══════════════════════════════════════════════════════════════════════
# CHIP 1  —  128M × 4 bits  (512 Mbits = 64 MB)
#
# Pinos do diagrama (b):
#   Endereço : A0-A12   (13 pinos, multiplexados via RAS / CAS)
#   Dados    : D0-D3    (4 pinos bidirecionais)
#   Controle : /RAS, /CAS, /CS, /WE, /OE
#   Banco    : Banco0, Banco1   (2 pinos de seleção)
#
# Organização interna:
#   2 bancos × 2^13 linhas × 2^13 colunas × 4 bits = 512 Mbits ✓
# ══════════════════════════════════════════════════════════════════════

class DRAM_128Mx4:                               # L01 — define a classe do chip

    # ── Constantes de hardware (existem em nível de classe) ──────────
    NUM_ADDR_PINS = 13    # L02 — 13 pinos de endereço: A0 a A12
    NUM_DATA_BITS = 4     # L03 — 4 bits por endereço:  D0 a D3
    NUM_BANKS     = 2     # L04 — 2 bancos: Banco0 e Banco1
    ROWS          = 1 << 13   # L05 — 2^13 = 8.192 linhas por banco
    COLS          = 1 << 13   # L06 — 2^13 = 8.192 colunas por banco
    # Total: 2 × 8192 × 8192 = 128 M endereços × 4 bits = 512 Mbits ✓

    def __init__(self):                          # L07 — construtor: inicializa o chip
        # Memória ESPARSA: dicionário em vez de lista para não alocar 64 MB reais.
        # Chave  → (banco, linha, coluna)
        # Valor  → nibble 0-15  (4 bits)
        self._mem        = {}    # L08 — dicionário esparso (só células escritas)
        self._row_latch  = None  # L09 — linha registrada quando /RAS↓ (nenhuma ainda)
        self._bank_latch = None  # L10 — banco registrado junto ao /RAS

    # ── Nível de sinal: ciclo RAS/CAS real ──────────────────────────

    def pulse_RAS(self, row, bank=0):            # L11 — simula borda ↓ de /RAS
        """
        Borda de descida de /RAS (Row Address Strobe).
        Hardware real: /RAS↓'congela' A0-A12 como endereço de LINHA
        e registra qual banco está ativo via Banco0/Banco1.
        """
        if not 0 <= bank < self.NUM_BANKS:       # L12 — banco deve ser 0 ou 1
            raise ValueError(f"Banco inválido: {bank}. Use 0 ou 1.")
        if not 0 <= row < self.ROWS:             # L13 — linha dentro do intervalo?
            raise ValueError(f"Linha {row} fora de [0, {self.ROWS-1}].")
        self._row_latch  = row   # L14 — trava a linha no registrador interno
        self._bank_latch = bank  # L15 — trava o banco no registrador interno

    def pulse_CAS_write(self, col, nibble):      # L16 — /CAS↓ com /WE ativo (escrita)
        """
        Borda de descida de /CAS com /WE=0 (Write Enable ativo).
        Hardware real: /CAS↓ reinterpreta A0-A12 como COLUNA;
        /WE=0 instrui o chip a copiar D0-D3 para a célula selecionada.
        """
        if self._row_latch is None:              # L17 — /RAS deve ter ocorrido antes
            raise RuntimeError("Chame pulse_RAS() antes de pulse_CAS_write().")
        if not 0 <= col < self.COLS:             # L18 — coluna dentro do intervalo?
            raise ValueError(f"Coluna {col} fora de [0, {self.COLS-1}].")
        if not 0 <= nibble <= 0xF:               # L19 — nibble cabe em 4 bits?
            raise ValueError(f"nibble deve estar em [0, 15]: recebeu {nibble}.")
        key = (self._bank_latch,                 # L20 — monta chave 3-D:
               self._row_latch, col)             #        (banco, linha, coluna)
        self._mem[key] = nibble                  # L21 — grava o nibble na memória

    def pulse_CAS_read(self, col):               # L22 — /CAS↓ com /OE ativo (leitura)
        """
        Borda de descida de /CAS com /OE=0 (Output Enable ativo).
        Hardware real: /OE=0 habilita os buffers dos pinos D0-D3 para saída;
        o conteúdo da célula selecionada aparece nos pinos de dado.
        """
        if self._row_latch is None:              # L23 — /RAS deve ter ocorrido antes
            raise RuntimeError("Chame pulse_RAS() antes de pulse_CAS_read().")
        if not 0 <= col < self.COLS:             # L24 — coluna dentro do intervalo?
            raise ValueError(f"Coluna {col} fora de [0, {self.COLS-1}].")
        key = (self._bank_latch,                 # L25 — reconstrói a chave 3-D
               self._row_latch, col)
        return self._mem.get(key, 0)             # L26 — lê (0 = célula virgem)

    # ── Interface de alto nível (abstrai RAS/CAS) ───────────────────

    def write(self, bank, row, col, nibble):     # L27 — escrita completa (simples)
        """Escreve 4 bits; executa RAS → CAS automaticamente."""
        self.pulse_RAS(row, bank)                # L28 — Passo 1: latcha linha/banco
        self.pulse_CAS_write(col, nibble)        # L29 — Passo 2: latcha coluna + grava

    def read(self, bank, row, col):              # L30 — leitura completa (simples)
        """Lê 4 bits; executa RAS → CAS automaticamente. Retorna int 0-15."""
        self.pulse_RAS(row, bank)                # L31 — Passo 1: latcha linha/banco
        return self.pulse_CAS_read(col)          # L32 — Passo 2: latcha coluna + lê

    def capacity_info(self):                     # L33 — exibe informações do chip
        total = self.NUM_BANKS * self.ROWS * self.COLS      # total de endereços
        bits  = total * self.NUM_DATA_BITS                  # total em bits
        return (
            f"DRAM 128M×4 | {self.NUM_BANKS}B × {self.ROWS}R × {self.COLS}C "
            f"= {total:,} end. × 4b "
            f"= {bits//(1<<20)} Mbits = {bits//(8<<20)} MB "
            f"| células escritas: {len(self._mem):,}"
        )


# ══════════════════════════════════════════════════════════════════════
# CHIP 2  —  4096K × 1 bit  (4 Mbits = 0,5 MB por chip)
#
# Pinos do diagrama (b):
#   Endereço : A0-A10  (11 pinos, multiplexados via RAS / CAS)
#   Dado     : D       (1 pino bidirecional)
#   Controle : /RAS, /CAS, /CS, /WE, /OE
#
# Nota do diagrama:
#   32 chips em paralelo → palavra de 32 bits, capacidade mínima 16 MB
# ══════════════════════════════════════════════════════════════════════

class DRAM_4096Kx1:                              # L34 — classe do chip individual

    NUM_ADDR_PINS = 11          # L35 — 11 pinos de endereço: A0 a A10
    ROWS          = 1 << 11     # L36 — 2^11 = 2.048 linhas
    COLS          = 1 << 11     # L37 — 2^11 = 2.048 colunas
    TOTAL_BITS    = ROWS * COLS # L38 — 2048 × 2048 = 4.194.304 bits = 4 Mbits

    def __init__(self, chip_id=0):               # L39 — chip_id identifica no sistema
        self._mem       = {}    # L40 — (linha, coluna) → 0 ou 1
        self._row_latch = None  # L41 — linha latchada pelo /RAS
        self.chip_id    = chip_id   # L42 — ID do chip no banco paralelo (0-31)

    # ── Nível de sinal ──────────────────────────────────────────────

    def pulse_RAS(self, row):                    # L43 — borda ↓ de /RAS
        """Latcha o endereço de linha."""
        if not 0 <= row < self.ROWS:             # L44 — valida intervalo
            raise ValueError(f"Linha {row} fora de [0, {self.ROWS-1}].")
        self._row_latch = row                    # L45 — registra a linha

    def pulse_CAS_write(self, col, bit):         # L46 — /CAS↓ + /WE → escrita
        """Grava 1 bit na célula (linha_latchada, col)."""
        if self._row_latch is None:              # L47 — /RAS deve ter ocorrido antes
            raise RuntimeError("Chame pulse_RAS() antes de pulse_CAS_write().")
        if not 0 <= col < self.COLS:             # L48 — valida coluna
            raise ValueError(f"Coluna {col} fora de [0, {self.COLS-1}].")
        if bit not in (0, 1):                    # L49 — dado deve ser exatamente 1 bit
            raise ValueError(f"bit deve ser 0 ou 1: recebeu {bit}.")
        self._mem[(self._row_latch, col)] = bit  # L50 — grava o bit

    def pulse_CAS_read(self, col):               # L51 — /CAS↓ + /OE → leitura
        """Lê e retorna 1 bit da célula (linha_latchada, col)."""
        if self._row_latch is None:              # L52
            raise RuntimeError("Chame pulse_RAS() antes de pulse_CAS_read().")
        if not 0 <= col < self.COLS:             # L53
            raise ValueError(f"Coluna {col} fora de [0, {self.COLS-1}].")
        return self._mem.get((self._row_latch, col), 0)  # L54 — lê (0 = virgem)

    # ── Interface de alto nível ─────────────────────────────────────

    def write(self, row, col, bit):              # L55
        """Escreve 1 bit em (linha, coluna). Gerencia RAS/CAS internamente."""
        self.pulse_RAS(row)                      # L56
        self.pulse_CAS_write(col, bit)           # L57

    def read(self, row, col):                    # L58
        """Lê 1 bit de (linha, coluna). Retorna 0 ou 1."""
        self.pulse_RAS(row)                      # L59
        return self.pulse_CAS_read(col)          # L60


# ══════════════════════════════════════════════════════════════════════
# SISTEMA  —  32 chips DRAM_4096Kx1 em paralelo
#
# Chip[i] armazena o bit i de cada palavra de 32 bits.
# Capacidade: 32 × 4 Mbits = 128 Mbits = 16 MB
# ══════════════════════════════════════════════════════════════════════

class Sistema32bits:                             # L61

    NUM_CHIPS = 32   # L62 — 32 chips = 32 bits por palavra

    def __init__(self):                          # L63
        # Instancia 32 chips; chip[i] é responsável pelo bit i de cada palavra
        self._chips = [DRAM_4096Kx1(chip_id=i)  # L64 — list comprehension: 32 chips
                       for i in range(self.NUM_CHIPS)]

    def write32(self, row, col, word):           # L65 — escreve 32 bits
        """
        Cada chip[i] recebe somente o bit i da palavra.
        No hardware real, todos os 32 chips são ativados simultaneamente (paralelo).
        """
        if not 0 <= word < (1 << 32):            # L66 — valida que cabe em 32 bits
            raise ValueError("Palavra deve ser uint32 [0, 2³²-1].")
        for i, chip in enumerate(self._chips):   # L67 — itera pelos 32 chips
            bit = (word >> i) & 1                # L68 — isola o bit na posição i
            chip.write(row, col, bit)            # L69 — chip[i] grava apenas esse bit

    def read32(self, row, col):                  # L70 — lê 32 bits
        """
        Cada chip[i] devolve seu 1 bit.
        A palavra de 32 bits é remontada por OR bit-a-bit com deslocamento.
        """
        word = 0                                 # L71 — acumulador da palavra
        for i, chip in enumerate(self._chips):   # L72 — itera pelos 32 chips
            bit  = chip.read(row, col)           # L73 — chip[i] retorna seu bit
            word |= (bit << i)                   # L74 — posiciona o bit na palavra
        return word                              # L75 — retorna a palavra completa

    def capacity_info(self):                     # L76
        mb = self.NUM_CHIPS * DRAM_4096Kx1.TOTAL_BITS // (8 * 1024 * 1024)
        return (
            f"Sistema {self.NUM_CHIPS} chips × 4 Mbits "
            f"= {self.NUM_CHIPS * 4} Mbits = {mb} MB "
            f"| palavra: {self.NUM_CHIPS} bits"
        )


# ══════════════════════════════════════════════════════════════════════
# DEMONSTRAÇÃO
# ══════════════════════════════════════════════════════════════════════

if __name__ == "__main__":

    LINE = "═" * 62

    # ── Chip 1 ──────────────────────────────────────────────────────
    print(LINE)
    print("  CHIP 1 — DRAM 128M × 4 bits  (512 Mbits = 64 MB)")
    print(LINE)
    chip1 = DRAM_128Mx4()
    print(chip1.capacity_info())
    print()

    # Escritas via interface de alto nível
    chip1.write(bank=0, row=100,  col=200,  nibble=0b1010)   # 1010b = 10
    chip1.write(bank=1, row=0,    col=0,    nibble=0xF)       # 0xF   = 15
    chip1.write(bank=0, row=8191, col=8191, nibble=5)         # posição máxima

    # Escrita via ciclo RAS/CAS explícito (nível de sinal)
    chip1.pulse_RAS(row=50, bank=0)
    chip1.pulse_CAS_write(col=75, nibble=7)

    # Leitura via ciclo RAS/CAS explícito
    chip1.pulse_RAS(row=50, bank=0)
    v_explícito = chip1.pulse_CAS_read(col=75)

    leituras = [
        ("banco=0 [100][200]",       chip1.read(0, 100,  200)),
        ("banco=1 [0][0]",           chip1.read(1, 0,    0)),
        ("banco=0 [8191][8191]",     chip1.read(0, 8191, 8191)),
        ("banco=0 [50][75] RAS/CAS", v_explícito),
    ]
    for label, v in leituras:
        print(f"  {label:33s} → {v:04b}b  ({v:2d})  0x{v:X}")

    print()
    print(chip1.capacity_info())

    # ── Chip 2 isolado ───────────────────────────────────────────────
    print()
    print(LINE)
    print("  CHIP 2 — DRAM 4096K × 1 bit  (4 Mbits = 0,5 MB/chip)")
    print(LINE)
    chip2 = DRAM_4096Kx1(chip_id=0)
    chip2.write(row=10,   col=20,   bit=1)
    chip2.write(row=10,   col=21,   bit=0)
    chip2.write(row=2047, col=2047, bit=1)   # endereço máximo

    for lbl, r, c in [("[10][20]", 10, 20),
                      ("[10][21]", 10, 21),
                      ("[2047][2047]", 2047, 2047)]:
        print(f"  {lbl:15s} → {chip2.read(r, c)}")

    # ── Sistema de 32 chips ──────────────────────────────────────────
    print()
    print(LINE)
    print("  SISTEMA — 32 chips DRAM_4096Kx1 em paralelo  (16 MB)")
    print(LINE)
    sys32 = Sistema32bits()
    print(sys32.capacity_info())
    print()

    tests = [
        (0, 0, 0x00000000),
        (0, 1, 0xDEADBEEF),
        (0, 2, 0x000000FF),
        (0, 3, 0xFFFFFFFF),
        (0, 4, 0x12345678),
    ]
    for r, c, w in tests:
        sys32.write32(r, c, w)

    print(f"  {'Endereço':15} {'Escrito':14} {'Lido':14} OK?")
    print("  " + "─" * 52)
    all_ok = True
    for r, c, w in tests:
        lido  = sys32.read32(r, c)
        ok    = (lido == w)
        all_ok = all_ok and ok
        print(f"  [{r},{c}]{'':<10} 0x{w:08X}   0x{lido:08X}   {'✓' if ok else '✗'}")
    print()
    print(f"  {'Todos os testes passaram! ✓' if all_ok else '⚠ FALHA!'}")
